# Random Grid Search

Baseline optimizer. See `docs/research.md` for methodology.

## Setup

In [1]:
# ruff: noqa: E402
import sys
from pathlib import Path

for _candidate in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
    if (_candidate / "prediction_market_extensions").is_dir() and (
        _candidate / "backtests"
    ).is_dir():
        if str(_candidate) not in sys.path:
            sys.path.insert(0, str(_candidate))
        break

from prediction_market_extensions.backtesting._notebook_support import ensure_notebook_repo_context

repo_root = ensure_notebook_repo_context()

## Configuration

In [6]:
from decimal import Decimal

from prediction_market_extensions.backtesting._execution_config import ExecutionModelConfig
from prediction_market_extensions.backtesting._execution_config import StaticLatencyConfig
from prediction_market_extensions.backtesting._experiments import ParameterSearchExperiment
from prediction_market_extensions.backtesting._prediction_market_runner import MarketDataConfig
from prediction_market_extensions.backtesting._replay_specs import QuoteReplay
from prediction_market_extensions.backtesting.data_sources import PMXT, Polymarket, QuoteTick
from prediction_market_extensions.backtesting.optimizers import ParameterSearchConfig
from prediction_market_extensions.backtesting.optimizers import ParameterSearchWindow


NAME = "notebook_optimizer"

DATA = MarketDataConfig(
    platform=Polymarket,
    data_type=QuoteTick,
    vendor=PMXT,
    sources=(
        "local:/Volumes/LaCie/pmxt_raws",
        "archive:r2.pmxt.dev",
        "relay:209-209-10-83.sslip.io",
    ),
)

BASE_REPLAY = QuoteReplay(
    market_slug="will-ludvig-aberg-win-the-2026-masters-tournament", token_index=0
)

TRAIN_WINDOWS = (
    ParameterSearchWindow(
        name="sample-a-full-window",
        start_time="2026-04-05T00:00:00Z",
        end_time="2026-04-07T23:59:59Z",
    ),
    ParameterSearchWindow(
        name="sample-b-2026-04-06-day",
        start_time="2026-04-06T00:00:00Z",
        end_time="2026-04-06T23:59:59Z",
    ),
    ParameterSearchWindow(
        name="sample-c-2026-04-07-late",
        start_time="2026-04-07T12:00:00Z",
        end_time="2026-04-07T23:59:59Z",
    ),
)

HOLDOUT_WINDOWS = (
    ParameterSearchWindow(
        name="sample-d-close-window",
        start_time="2026-04-07T00:00:00Z",
        end_time="2026-04-07T11:59:59Z",
    ),
)

STRATEGY_SPEC = {
    "strategy_path": "strategies:QuoteTickEMACrossoverStrategy",
    "config_path": "strategies:QuoteTickEMACrossoverConfig",
    "config": {
        "trade_size": Decimal("5"),
        "fast_period": "__SEARCH__:fast_period",
        "slow_period": "__SEARCH__:slow_period",
        "entry_buffer": "__SEARCH__:entry_buffer",
        "take_profit": "__SEARCH__:take_profit",
        "stop_loss": "__SEARCH__:stop_loss",
    },
}

PARAMETER_GRID = {
    "fast_period": (32, 64, 96),
    "slow_period": (128, 256, 384),
    "entry_buffer": (0.00025, 0.0005),
    "take_profit": (0.005, 0.01),
    "stop_loss": (0.005, 0.01),
}

EXECUTION = ExecutionModelConfig(
    queue_position=True,
    latency_model=StaticLatencyConfig(
        base_latency_ms=75.0,
        insert_latency_ms=10.0,
        update_latency_ms=5.0,
        cancel_latency_ms=5.0,
    ),
)

MAX_TRIALS = 18
HOLDOUT_TOP_K = 5

PARAMETER_SEARCH = ParameterSearchConfig(
    name=NAME,
    data=DATA,
    base_replay=BASE_REPLAY,
    strategy_spec=STRATEGY_SPEC,
    parameter_grid=PARAMETER_GRID,
    train_windows=TRAIN_WINDOWS,
    holdout_windows=HOLDOUT_WINDOWS,
    max_trials=MAX_TRIALS,
    random_seed=7,
    holdout_top_k=HOLDOUT_TOP_K,
    initial_cash=100.0,
    probability_window=256,
    min_quotes=500,
    min_price_range=0.005,
    min_fills_per_window=1,
    execution=EXECUTION,
    emit_html=False,
    chart_output_path="output",
    sampler="random",
)

EXPERIMENT = ParameterSearchExperiment(
    name=NAME,
    description="EMA random-grid search on PMXT L2 data",
    parameter_search=PARAMETER_SEARCH,
)

print(
    f"Configured random-grid search: {MAX_TRIALS} trials over "
    f"{len(TRAIN_WINDOWS)} train window(s), {len(HOLDOUT_WINDOWS)} holdout window(s)."
)

Configured random-grid search: 18 trials over 3 train window(s), 1 holdout window(s).


## Run optimizer

In [3]:
from pprint import pprint

from prediction_market_extensions.backtesting._experiments import run_experiment_async
from prediction_market_extensions.backtesting._notebook_support import suppress_notebook_cell_output

with suppress_notebook_cell_output():
    summary = await run_experiment_async(EXPERIMENT)

print("Selected params:")
pprint(dict(summary.selected_params))
print("Leaderboard CSV:", summary.leaderboard_csv_path)
print("Summary JSON:", summary.summary_json_path)

Selected params:
{'entry_buffer': 0.0005,
 'fast_period': 64,
 'slow_period': 384,
 'stop_loss': 0.01,
 'take_profit': 0.01}
Leaderboard CSV: /Users/evankolberg/prediction-market-backtesting/output/notebook_optimizer_leaderboard.csv
Summary JSON: /Users/evankolberg/prediction-market-backtesting/output/notebook_optimizer_summary.json


## Leaderboard

In [4]:
import pandas as pd
from IPython.display import display, Markdown

_leaderboard_df = pd.read_csv(summary.leaderboard_csv_path)
_display_cols = [
    c
    for c in (
        "trial_id",
        "train_median_score",
        "holdout_median_score",
        "train_median_pnl",
        "holdout_median_pnl",
        "train_median_drawdown",
        "train_median_fills",
        "params_json",
    )
    if c in _leaderboard_df.columns
]
_top = _leaderboard_df[_display_cols].head(10).copy()

_fmt = {
    "train_median_score": "{:.4f}".format,
    "holdout_median_score": "{:.4f}".format,
    "train_median_pnl": "{:.4f}".format,
    "holdout_median_pnl": "{:.4f}".format,
    "train_median_drawdown": "{:.4f}".format,
    "train_median_fills": "{:.1f}".format,
}
_fmt = {k: v for k, v in _fmt.items() if k in _top.columns}

_styled = (
    _top.style.format(_fmt, na_rep="-")
    .hide(axis="index")
    .set_caption(f"Top candidates — {EXPERIMENT.name} (random-grid)")
)
display(_styled)

display(
    Markdown(
        "### Selected parameters\n\n"
        + "\n".join(f"- **{k}**: `{v}`" for k, v in dict(summary.selected_params).items())
    )
)

trial_id,train_median_score,holdout_median_score,train_median_pnl,holdout_median_pnl,train_median_drawdown,train_median_fills,params_json
13,-0.4375,-0.4250,-0.1050,-0.1050,0.6650,14.0,"{""entry_buffer"": 0.0005, ""fast_period"": 64, ""slow_period"": 384, ""stop_loss"": 0.01, ""take_profit"": 0.01}"
10,-0.3625,-0.4775,-0.0650,-0.1400,0.6200,9.0,"{""entry_buffer"": 0.00025, ""fast_period"": 64, ""slow_period"": 128, ""stop_loss"": 0.005, ""take_profit"": 0.005}"
11,-0.4425,-0.5425,-0.1150,-0.1800,0.6750,18.0,"{""entry_buffer"": 0.00025, ""fast_period"": 96, ""slow_period"": 256, ""stop_loss"": 0.005, ""take_profit"": 0.01}"
12,-0.4425,-0.5425,-0.1150,-0.1800,0.6750,18.0,"{""entry_buffer"": 0.00025, ""fast_period"": 96, ""slow_period"": 256, ""stop_loss"": 0.005, ""take_profit"": 0.005}"
17,-0.4125,-0.5850,-0.1000,-0.2100,0.6550,16.0,"{""entry_buffer"": 0.0005, ""fast_period"": 32, ""slow_period"": 256, ""stop_loss"": 0.01, ""take_profit"": 0.005}"
18,-0.4425,-,-0.1050,-,0.6750,16.0,"{""entry_buffer"": 0.00025, ""fast_period"": 96, ""slow_period"": 256, ""stop_loss"": 0.01, ""take_profit"": 0.005}"
6,-0.4500,-,-0.1100,-,0.6800,18.0,"{""entry_buffer"": 0.0005, ""fast_period"": 32, ""slow_period"": 384, ""stop_loss"": 0.01, ""take_profit"": 0.01}"
9,-0.4500,-,-0.1100,-,0.6800,18.0,"{""entry_buffer"": 0.0005, ""fast_period"": 32, ""slow_period"": 384, ""stop_loss"": 0.01, ""take_profit"": 0.005}"
4,-0.4950,-,-0.1400,-,0.7100,22.0,"{""entry_buffer"": 0.00025, ""fast_period"": 96, ""slow_period"": 384, ""stop_loss"": 0.005, ""take_profit"": 0.01}"
3,-0.5025,-,-0.1450,-,0.7150,24.0,"{""entry_buffer"": 0.00025, ""fast_period"": 64, ""slow_period"": 256, ""stop_loss"": 0.005, ""take_profit"": 0.005}"


### Selected parameters

- **fast_period**: `64`
- **slow_period**: `384`
- **entry_buffer**: `0.0005`
- **take_profit**: `0.01`
- **stop_loss**: `0.01`

## Holdout replay

In [ ]:
import gc

from prediction_market_extensions.backtesting._notebook_support import (
    display_html_artifacts,
    find_updated_html_artifacts,
    snapshot_html_artifacts,
    suppress_notebook_cell_output,
)
from prediction_market_extensions.backtesting.optimizers import (
    build_parameter_search_window_backtest,
)

selected_window = HOLDOUT_WINDOWS[0] if HOLDOUT_WINDOWS else TRAIN_WINDOWS[-1]

output_root = repo_root / "output"
html_snapshot = snapshot_html_artifacts(output_root)

backtest = build_parameter_search_window_backtest(
    config=PARAMETER_SEARCH,
    window=selected_window,
    params=summary.selected_params,
    trial_id=0,
    name=f"{NAME}:{selected_window.name}:selected-params",
    emit_html=True,
    chart_output_path=str(output_root),
    return_summary_series=False,
)

with suppress_notebook_cell_output():
    results = await backtest.run_async()

updated_html = find_updated_html_artifacts(output_root, html_snapshot)
print(f"Replayed {len(results)} market(s) on window '{selected_window.name}'.")
display_html_artifacts(updated_html, repo_root=repo_root)

# Drop refs and GC so re-running this cell does not leak replay state into
# the kernel — each re-run otherwise adds ~500 MB and macOS will SIGKILL
# the kernel after a few iterations.
del results, backtest, updated_html, html_snapshot
gc.collect()